# Streaming / Real-Time Analysis with tttrlib

This notebook demonstrates tttrlib's streaming consumers for **live photon stream analysis**. All four consumers accept photons one at a time and maintain incremental state.

- `StreamingCorrelator` — online multi-tau FCS correlation
- `StreamingBurstDetector` — sliding-window burst search
- `StreamingDecayHistogram` — incremental TCSPC decay histogram
- `StreamingPhasor` — incremental FLIM phasor (g, s)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tttrlib

rng = np.random.default_rng(42)

## 1. Streaming FCS Correlation

Simulate 3D Brownian diffusion through a confocal Gaussian volume, then stream photons one at a time. The resulting G(τ) shows a characteristic diffusion decay.

In [ ]:
# Confocal volume parameters
w_xy = 0.5e-6    # lateral waist (m)
w_z  = 2.0e-6    # axial waist (m)
D    = 4e-12     # diffusion coefficient (m²/s)
tau_D = w_xy**2 / (4 * D)

# Simulation grid
dt = tau_D / 50
n_steps = 200_000
box = 8e-6
n_particles = 4
count_rate_per_particle = 3.0

print(f"τ_D = {tau_D*1e3:.2f} ms, dt = {dt*1e6:.1f} µs")

# Brownian diffusion simulation
pos = rng.uniform(-box/2, box/2, size=(n_particles, 3))
brownian = np.sqrt(2 * D * dt)
photon_times = []

for step in range(n_steps):
    pos += brownian * rng.standard_normal((n_particles, 3))
    pos = np.mod(pos + box/2, box) - box/2  # periodic boundary
    r2_xy = pos[:, 0]**2 + pos[:, 1]**2
    z2 = pos[:, 2]**2
    brightness = np.exp(-2*r2_xy/w_xy**2) * np.exp(-2*z2/w_z**2)
    n_ph = rng.poisson(count_rate_per_particle * np.sum(brightness))
    if n_ph > 0:
        photon_times.extend([step] * n_ph)

photon_times = np.array(photon_times, dtype=np.uint64)
print(f"Generated {len(photon_times)} photons ({len(photon_times)/n_steps/dt/1e3:.1f} kHz)")

In [ ]:
# Stream photons one at a time
corr = tttrlib.StreamingCorrelator(n_bins=16, n_casc=25, macro_time_resolution=dt)

snapshots = []
cps = [len(photon_times)//4, len(photon_times)//2, 3*len(photon_times)//4, len(photon_times)]
for i, t in enumerate(photon_times):
    corr.push_photon(int(t))
    if i + 1 in cps:
        snapshots.append((i+1, corr.x_axis.copy(), corr.correlation_normalized.copy()))

print(f"Streaming: {corr.photon_count()} photons")

In [ ]:
# Plot evolving + final with theoretical overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: snapshots
for n, x, g in snapshots:
    m = x > 0
    axes[0].semilogx(x[m]*dt*1e3, g[m]-1, label=f"N={n:,}", alpha=0.8)
axes[0].axvline(tau_D*1e3, color='k', ls=':', alpha=0.4)
axes[0].set_xlabel("τ (ms)"); axes[0].set_ylabel("G(τ) - 1")
axes[0].set_title("Building up"); axes[0].legend(fontsize=8); axes[0].set_ylim(bottom=0)

# Right: final + theory
x = corr.x_axis; g = corr.correlation_normalized
m = x > 0; tau = x[m] * dt
axes[1].semilogx(tau*1e3, g[m]-1, 'b-', alpha=0.8, label="Streaming")
kappa = w_z / w_xy
vol = np.pi**1.5 * w_xy**2 * w_z
N_eff = n_particles * vol / (box**3)
G_th = (1/N_eff) * (1 + tau/tau_D)**(-1) * (1 + tau/(tau_D*kappa**2))**(-0.5)
axes[1].semilogx(tau*1e3, G_th, 'r--', alpha=0.8, label=f"Theory (N={N_eff:.1f})")
axes[1].axvline(tau_D*1e3, color='k', ls=':', alpha=0.4)
axes[1].set_xlabel("τ (ms)"); axes[1].set_ylabel("G(τ) - 1")
axes[1].set_title("Final curve"); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

plt.tight_layout(); plt.show()

## 2. Streaming Burst Detection

Sliding-window burst search: detects bursts as photons arrive.

In [ ]:
# Bursty trace: molecules passing through volume
times_b = []
state = False
t = 0
for _ in range(30_000):
    rate = 0.5 if state else 0.01
    t += int(rng.exponential(1.0/rate)) + 1
    times_b.append(t)
    if rng.random() < (0.0005 if state else 0.001):
        state = not state
times_b = np.array(times_b, dtype=np.uint64)

det = tttrlib.StreamingBurstDetector(window_photons=10, window_time=100.0)
det.push_np(times_b)
det.flush()
bursts = det.bursts
n_bursts = len(bursts) // 2
print(f"Detected {n_bursts} bursts out of {det.photon_count()} photons")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
h, e = np.histogram(times_b, bins=200)
ax.fill_between(0.5*(e[:-1]+e[1:]), h, alpha=0.5)
for i in range(n_bursts):
    ax.axvspan(times_b[bursts[2*i]], times_b[bursts[2*i+1]], alpha=0.15, color='red')
ax.set_xlabel("Macro time"); ax.set_ylabel("Photons")
ax.set_title(f"Burst detection ({n_bursts} bursts)")
plt.tight_layout(); plt.show()

## 3. Streaming Decay Histogram + FLIM Phasor

Incremental TCSPC histogram and phasor computation.

In [ ]:
# Single-exponential decay (τ = 3 ns at 80 MHz)
n_bins = 4096; freq = 80.0
dt_ns = 1e3 / freq; tau = 3.0
bin_t = np.arange(n_bins) * dt_ns
pdf = np.exp(-bin_t / tau); pdf /= pdf.sum()
n_ph = 50_000
mts = rng.choice(n_bins, size=n_ph, p=pdf).astype(np.uint16)

# Streaming histogram
hist = tttrlib.StreamingDecayHistogram(n_microtime_bins=n_bins, n_channels=1)
hist.push_np(mts)
decay = np.array(hist.get_histogram(0))

# Streaming phasor
phasor = tttrlib.StreamingPhasor(freq, n_bins, dt_ns*1e-9)
phasor.push_np(mts)
g, s, n = phasor.get_phasor()

# Theory
omega = 2*np.pi*freq*1e6*tau*1e-9
g_th = 1/(1+omega**2); s_th = omega/(1+omega**2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(bin_t, decay); axes[0].set_xlim(0, 12)
axes[0].set_xlabel("Time (ns)"); axes[0].set_ylabel("Counts")
axes[0].set_title("Decay histogram")

th = np.linspace(0, np.pi/2, 100)
axes[1].plot(0.5+0.5*np.cos(th), 0.5*np.sin(th), 'k--', alpha=0.3)
axes[1].plot(g_th, s_th, 'r+', ms=15, label=f"Theory τ={tau} ns")
axes[1].plot(g, s, 'bo', label=f"Streaming ({n:.0f})")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 0.7); axes[1].set_aspect('equal')
axes[1].set_xlabel("g"); axes[1].set_ylabel("s")
axes[1].set_title("Phasor"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Phasor:  g={g:.4f}, s={s:.4f}")
print(f"Theory:  g={g_th:.4f}, s={s_th:.4f}")

## 4. Combined Live Session

All four consumers running simultaneously on the same photon stream — a realistic live experiment.

In [ ]:
corr = tttrlib.StreamingCorrelator(16, 25, dt)
det = tttrlib.StreamingBurstDetector(10, 100.0)
hist2 = tttrlib.StreamingDecayHistogram(256, 1)
phasor2 = tttrlib.StreamingPhasor(80.0, 256, dt_ns*1e-9)

for i in range(min(20_000, len(photon_times))):
    mt = int(photon_times[i])
    mt_fine = int(mts[i % len(mts)])
    corr.push_photon(mt)
    det.push_photon(mt)
    hist2.push_photon(mt_fine, 0)
    phasor2.push_photon(mt_fine)

print(f"Correlator: {corr.photon_count()} photons")
print(f"Bursts:     {det.burst_count} detected")
print(f"Decay:      {hist2.total_count()} photons")
g2, s2, n2 = phasor2.get_phasor()
print(f"Phasor:     g={g2:.4f}, s={s2:.4f}, n={n2:.0f}")